# pyGeoMapper — Notebook 2: Multiscale 2D CWT Terrain Complexity & Landform Scale Analysis

**Author:** Larry Syu-Heng Lai  
**License:** AGPL-3.0-or-later (`pyGeoMapper` is an independent Rust + Python geoscience toolkit and is distinct from `The-Magicians-Code/PyGeoMapper`.)

This notebook demonstrates **`terrain_analysis`** of `pyGeoMapper`, implementing a parallel Rust **2D Continuous Wavelet Transform (`2D CWT`)** with the isotropic Mexican-hat (Marr / Ricker) wavelet across spatial scales $s \in [20\text{ m}, 240\text{ m}]$:
$$\psi(x, y; s) = \frac{1}{\pi s^2}\left(1 - \frac{x^2 + y^2}{2s^2}\right)\exp\!\left(-\frac{x^2 + y^2}{2s^2}\right), \qquad \lambda(s) = \sqrt{2}\,\pi s$$

- **Methodological Citation:** Adapted from **pyTopoComplexity** (Lai, S.-H., et al., 2025, *Earth Surface Dynamics*, 13, 417–438, https://doi.org/10.5194/esurf-13-417-2025). If `pytopocomplexity` is installed in the Python environment, `terrain_analysis.run_pytopocomplexity()` can invoke it directly while defaulting seamlessly to the parallel Rust kernel (`_core.compute_multiscale_complexity_rs`).
- **Case Study 1 (Tectonic Hogbacks & Folded Bedrock):** **Sheep Mountain Anticline, Bighorn Basin, Wyoming** (`EPSG:32612`, USGS 3DEP DEM `20 m` grid) — multiscale wavelet energy isolates fine inner-canyon incisions (`s = 30 m`) vs. broad anticlinal hogback ridges (`s = 120–240 m`).
- **Case Study 2 (Catastrophic Landslide Roughness Change):** **2014 Oso (SR 530) Landslide, Washington** (`EPSG:32610`, pre-event Copernicus GLO-30 vs. post-event USGS 3DEP LiDAR DEM) — quantifying short-wavelength hummocky debris-avalanche complexity enhancement alongside 2D, 2.5D/3D, and swath cross-profile views.

### About this notebook
- **Purpose:** multiscale 2D CWT terrain complexity on Sheep Mountain (WY) and roughness change across the 2014 Oso landslide (WA).
- **Data used:** `example_data/sheep_mtn_dem_3dep.tif`, `sheep_mtn_geol_units.tif`, `sheep_mtn_units.json`, `oso_dem_pre_cop30.tif`, `oso_dem_post_3dep.tif`.
- **Figures written:** `image/oso_cwt_change.png` (README), `image/cwt_multiscale.png` (appendix board), `image/sheep_mtn_cwt_3d_drape.png`.
- **How to re-run:** run all cells from `notebooks/` or the repo root. Figures are saved to `image/` with `plotting.save_figure` and are not embedded in the notebook.

In [1]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

from pygeomapper import (
    raster_io,
    terrain_analysis,
    cross_section,
    viewer_3d,
    plotting,
)

ROOT = Path.cwd() if (Path.cwd() / "example_data").exists() else Path.cwd().parent   # repo root (run from notebooks/ or the root)

plt.rcParams['figure.dpi'] = 120
DATA_DIR = ROOT / "example_data"
IMG_DIR = ROOT / "image"
IMG_DIR.mkdir(exist_ok=True)

sheep_dem = raster_io.load_geotiff(DATA_DIR / 'sheep_mtn_dem_3dep.tif', name='sheep_dem_3dep', units='m')
sheep_geol = raster_io.load_geotiff(DATA_DIR / 'sheep_mtn_geol_units.tif', name='sheep_geol_units', units='unit_id')
oso_pre = raster_io.load_geotiff(DATA_DIR / 'oso_dem_pre_cop30.tif', name='oso_dem_pre', units='m')
oso_post = raster_io.load_geotiff(DATA_DIR / 'oso_dem_post_3dep.tif', name='oso_dem_post', units='m')

print(f'Sheep Mountain 3DEP DEM: {sheep_dem.shape}, CRS={sheep_dem.crs}, relief={np.nanmin(sheep_dem.data):.1f}..{np.nanmax(sheep_dem.data):.1f} m')
print(f'Oso Pre/Post DEMs:       {oso_post.shape}, CRS={oso_post.crs}, relief={np.nanmin(oso_post.data):.1f}..{np.nanmax(oso_post.data):.1f} m')

Sheep Mountain 3DEP DEM: (160, 160), CRS=EPSG:32612, relief=1134.0..1512.8 m
Oso Pre/Post DEMs:       (200, 200), CRS=EPSG:32610, relief=71.8..277.8 m


## 1. Synthetic Wavelength Recovery Verification (2D Mexican-Hat CWT)
Before analyzing real terrain, we verify that the Rust separable 2D Mexican-hat CWT (`compute_multiscale_complexity`) recovers the exact theoretical wavelength $\lambda_0 = 120\text{ m}$ ($s_0 = \lambda_0 / (\sqrt{2}\pi) \approx 27.01\text{ m}$) of a 2D sinusoidal ridge-and-valley surface.

In [2]:
xs_syn = np.arange(128, dtype=np.float64) * 5.0
ys_syn = np.arange(128, dtype=np.float64) * 5.0
xx, yy = np.meshgrid(xs_syn, ys_syn)
target_lambda_m = 120.0
syn_surface = 25.0 * np.sin(2.0 * np.pi * xx / target_lambda_m) + 15.0 * np.cos(2.0 * np.pi * yy / target_lambda_m)

scales_test = [12.0, 18.0, 27.0, 40.0, 60.0]
syn_cwt = terrain_analysis.compute_multiscale_complexity(
    syn_surface, scales_m=scales_test, pixel_size_m=5.0
)
core_wl = syn_cwt['dominant_wavelength_m'][25:-25, 25:-25]
median_wl = float(np.nanmedian(core_wl))
rel_err_pct = 100.0 * abs(median_wl - target_lambda_m) / target_lambda_m
print(f'Target wavelength:    {target_lambda_m:.2f} m')
print(f'Recovered wavelength: {median_wl:.2f} m (relative error = {rel_err_pct:.2f}%)')

Target wavelength:    120.00 m
Recovered wavelength: 116.84 m (relative error = 2.64%)


## 2. Multiscale 2D CWT Terrain Complexity over Sheep Mountain Anticline, WY

### 1. What This Step Does & Why We Use This Method
- Applies the parallel Rust 2D Continuous Wavelet Transform (`compute_multiscale_complexity` / `run_pytopocomplexity`, adapted from *pyTopoComplexity*, Lai et al., 2025, *ESurf*) across spatial scales $s \in \{30, 60, 120, 220\}\text{ m}$.
- Unlike standard $3\times 3$ slope or curvature (which only measure local grid-cell derivatives), 2D CWT decomposes topography by **characteristic spatial wavelength** $\lambda(s) = \sqrt{2}\pi s$.

### 2. Governing Equations & Physical Meaning
$$W(x, y; s) = (z * \psi_s)(x, y), \qquad \psi_s(x, y) = \frac{1}{\pi s^2}\left(1 - \frac{x^2 + y^2}{2s^2}\right)\exp\!\left(-\frac{x^2 + y^2}{2s^2}\right)$$
$$E(x, y; s) = \frac{|W(x, y; s)|}{s}, \qquad \lambda_{\text{dom}}(x, y) = \sqrt{2}\pi\,\arg\max_{s} E(x, y; s)$$
- **Fine scale ($s = 30\text{ m}$, $\lambda \approx 133\text{ m}$):** Isolates steep inner-canyon walls along the Bighorn River gorge and sharp flatiron/hogback crests.
- **Coarse scale ($s = 120\text{–}220\text{ m}$, $\lambda \approx 533\text{–}977\text{ m}$):** Captures the macroscopic fold ridge-and-strike-valley wavelength.


In [3]:
sheep_scales_m = [30.0, 60.0, 120.0, 220.0]
sheep_cwt = terrain_analysis.run_pytopocomplexity(sheep_dem, scales_m=sheep_scales_m)
sheep_hs = terrain_analysis.compute_hillshade(sheep_dem, azimuth_deg=315.0, altitude_deg=42.0)

# Summarize mean multiscale complexity by USGS SGMC bedrock formation
meta = json.loads((DATA_DIR / 'sheep_mtn_units.json').read_text())
print(f'Backend used: {sheep_cwt["backend"]} ({sheep_cwt["citation"]})\n')
print(f'{"Unit":6s} | {"Formation Name":32s} | {"Mean Complexity":>15s} | {"Median Wavelength (m)":>21s}')
print('-' * 84)
for u in meta['units']:
    uid = u['unit_id']
    m = sheep_geol.data == uid
    mc = float(np.nanmean(sheep_cwt['mean_complexity'].data[m]))
    mw = float(np.nanmedian(sheep_cwt['dominant_wavelength_m'].data[m]))
    print(f"{u['code']:6s} | {u['name']:32s} | {mc:15.2f} | {mw:21.1f}")

Backend used: pytopocomplexity (Lai, L. S.-H., et al. (2025). pyTopoComplexity: multiscale 2D continuous wavelet terrain complexity. Earth Surface Dynamics, 13, 417-438, https://doi.org/10.5194/esurf-13-417-2025)

Unit   | Formation Name                   | Mean Complexity | Median Wavelength (m)
------------------------------------------------------------------------------------
Mm     | Madison Limestone                |           27.24 |                 977.4
IPta   | Tensleep Sandstone & Amsden Fm   |           18.02 |                 977.4
TRPc   | Chugwater & Goose Egg Fms        |           24.08 |                 977.4
Jsg    | Sundance & Gypsum Spring Fms     |           16.31 |                 977.4
KJcm   | Cloverly & Morrison Fms          |            9.54 |                 977.4
Kmt    | Mowry & Thermopolis Shales       |            4.63 |                 977.4
Qa     | Quaternary Alluvium & Terrace    |            8.49 |                 977.4


## 3. Pre- vs. Post-Landslide Multiscale Complexity Change (2014 Oso Landslide, WA)
Next we apply `terrain_analysis.compare_complexity_change(oso_pre, oso_post, scales_m=[15, 30, 60, 120])` to quantify how the March 22, 2014 Oso landslide altered surface roughness and characteristic landform wavelength across the headscarp and hummocky debris-avalanche deposit.

In [4]:
oso_scales_m = [15.0, 30.0, 60.0, 120.0]
oso_comp = terrain_analysis.compare_complexity_change(oso_pre, oso_post, scales_m=oso_scales_m)

delta_c = oso_comp['delta_complexity'].data
ratio_c = oso_comp['complexity_ratio'].data
print(f'Oso Complexity Change (Post - Pre): min={np.nanmin(delta_c):.2f}, max=+{np.nanmax(delta_c):.2f}')
print(f'Oso Complexity Ratio  (Post / Pre): median={np.nanmedian(ratio_c):.2f}x, 95th pct={np.nanpercentile(ratio_c, 95):.2f}x')

Oso Complexity Change (Post - Pre): min=-5.73, max=+1.49
Oso Complexity Ratio  (Post / Pre): median=1.00x, 95th pct=1.04x


## 4. Oso Landslide Roughness Change: Pre → Post → Δ (`image/oso_cwt_change.png`)
One clean 3-panel figure of the multiscale CWT complexity before and after the 2014 Oso landslide, and their difference.

### How to Read the Visual Output
1. **Panel (a) — Pre-slide complexity** from the Copernicus GLO-30 DEM (2011–2013). Bright = rough terrain at 15–120 m scales.
2. **Panel (b) — Post-slide complexity** from the USGS 3DEP lidar DEM, same color scale as (a).
3. **Panel (c) — Change Δ = post − pre:** orange = rougher after the slide, purple = smoother.
- **Caveat:** GLO-30 is a radar surface model (it includes forest canopy) while 3DEP is bare-earth lidar, so part of the change is canopy vs. ground surface, not only landslide reshaping.

In [5]:
c_pre = oso_comp["t1"]["mean_complexity"].data
c_post = oso_comp["t2"]["mean_complexity"].data
oso_hs_post = terrain_analysis.compute_hillshade(oso_post)
ext_oso = oso_post.extent
vmax_c = float(np.nanpercentile(np.r_[c_pre.ravel(), c_post.ravel()], 98))
vlim_d = float(np.nanpercentile(np.abs(delta_c), 98))

fig, axes = plt.subplots(1, 3, figsize=(15, 5.0), layout="constrained")
for ax, arr, title in [
    (axes[0], c_pre, "a) Pre-slide complexity (GLO-30, 2011–13)"),
    (axes[1], c_post, "b) Post-slide complexity (3DEP lidar)"),
]:
    ax.imshow(oso_hs_post.data, cmap="gray", extent=ext_oso, origin="upper")
    im_c = ax.imshow(arr, cmap="magma", vmin=0.0, vmax=vmax_c, alpha=0.80, extent=ext_oso, origin="upper")
    ax.set_title(title, fontweight="bold")
fig.colorbar(im_c, ax=axes[:2], shrink=0.85, label="Mean CWT complexity |W(s)|/s (s = 15–120 m)")
axes[2].imshow(oso_hs_post.data, cmap="gray", extent=ext_oso, origin="upper")
im_d = axes[2].imshow(delta_c, cmap="PuOr_r", vmin=-vlim_d, vmax=vlim_d, alpha=0.80, extent=ext_oso, origin="upper")
axes[2].set_title("c) Change (post − pre)\norange = rougher, purple = smoother", fontweight="bold")
fig.colorbar(im_d, ax=axes[2], shrink=0.85, label="Δ complexity")
for ax in axes:
    plotting.use_plain_axes(ax, unit="km", xlabel="Easting", ylabel="Northing")
fig.suptitle("2014 Oso landslide — multiscale terrain roughness before and after (Rust 2D CWT)", fontweight="bold")
out_oso = plotting.save_figure(fig, IMG_DIR / "oso_cwt_change.png", max_width_px=1800)
print("Saved:", out_oso)

Saved: image/oso_cwt_change.png


## 5. Appendix: 6-Panel CWT Summary Board (`image/cwt_multiscale.png`)

### How to Read the 6-Panel Visual Output
1. **Panel (a) — 3DEP Hillshade & Contours:** Shaded relief of Sheep Mountain Anticline cut by the superposed Bighorn River canyon (`A → A'` swath line).
2. **Panel (b) — Signed Fine-Scale Wavelet Response $W(x,y; s=30\text{ m})$:** Red (`W > 0`) marks sharp convex ridges/hogback crests; blue (`W < 0`) marks incised canyon bottoms and strike valleys.
3. **Panel (c) — Broad-Scale Wavelet Energy $|W(x,y; s=120\text{ m})|/s$:** Bright gold/white (`magma`) highlights the high-relief anticlinal core and major structural hogbacks.
4. **Panel (d) — Dominant Landform Wavelength $\lambda_{\text{dom}}(x,y)$:** Shows spatial partitioning between short-wavelength badland/gully dissection (purple/blue) and long-wavelength anticlinal limbs (green/yellow).
5. **Panel (e) — Oso Landslide Complexity Change $\Delta C = C_{\text{post}} - C_{\text{pre}}$:** orange = rougher after the slide, purple = smoother (same data as Section 4, panel c).
6. **Panel (f) — Swath Cross-Profile (`A → A'`):** Compares topographic elevation $z(s)$ against fine-scale ($s=30\text{ m}$) and broad-scale ($s=120\text{ m}$) wavelet energy peaks.


In [6]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10), layout="constrained")
ext_s = [sheep_dem.bounds[0]/1e3, sheep_dem.bounds[2]/1e3, sheep_dem.bounds[1]/1e3, sheep_dem.bounds[3]/1e3]

# Panel (a): Sheep Mountain 3DEP DEM + Hillshade + Transect
ax = axes[0, 0]
ax.imshow(sheep_hs.data, cmap='gray', extent=ext_s, origin='upper')
im0 = ax.imshow(sheep_dem.data, cmap='terrain', alpha=0.58, extent=ext_s, origin='upper')
p_start, p_end = (726900.0, 4942800.0), (729700.0, 4944600.0)
ax.plot([p_start[0]/1e3, p_end[0]/1e3], [p_start[1]/1e3, p_end[1]/1e3], 'r--', lw=2.0, label='Swath A–A\'')
ax.set_title('(a) Sheep Mountain DEM + hillshade', fontweight='bold')
ax.set_xlabel('UTM Easting (km, Zone 12N)')
ax.set_ylabel('UTM Northing (km)')
ax.legend(loc='lower right', fontsize=8)
plt.colorbar(im0, ax=ax, fraction=0.046, pad=0.04, label='Elevation (m)')

# Panel (b): 2D Mexican-Hat Wavelet Response W(x, y; s=30 m)
ax = axes[0, 1]
cwt_30 = sheep_cwt['cwt_stack'].band('cwt_30m').data
vmax_w = float(np.nanpercentile(np.abs(cwt_30), 98))
im1 = ax.imshow(cwt_30, cmap='RdBu_r', vmin=-vmax_w, vmax=vmax_w, extent=ext_s, origin='upper')
ax.set_title('(b) CWT response W (s = 30 m): ridges + / valleys −', fontweight='bold')
ax.set_xlabel('UTM Easting (km, Zone 12N)')
ax.set_ylabel('UTM Northing (km)')
plt.colorbar(im1, ax=ax, fraction=0.046, pad=0.04, label='Wavelet Amplitude (m)')

# Panel (c): Multiscale Mean Wavelet Complexity (Sheep Mountain)
ax = axes[0, 2]
im2 = ax.imshow(sheep_cwt['mean_complexity'].data, cmap='magma', extent=ext_s, origin='upper')
ax.set_title('(c) Mean complexity (s = 30–220 m)', fontweight='bold')
ax.set_xlabel('UTM Easting (km, Zone 12N)')
ax.set_ylabel('UTM Northing (km)')
plt.colorbar(im2, ax=ax, fraction=0.046, pad=0.04, label='Mean |W(s)| / s')

# Panel (d): Dominant Characteristic Wavelength Map (Sheep Mountain)
ax = axes[1, 0]
im3 = ax.imshow(sheep_cwt['dominant_wavelength_m'].data, cmap='viridis', extent=ext_s, origin='upper')
ax.set_title('(d) Dominant wavelength λ_dom', fontweight='bold')
ax.set_xlabel('UTM Easting (km, Zone 12N)')
ax.set_ylabel('UTM Northing (km)')
plt.colorbar(im3, ax=ax, fraction=0.046, pad=0.04, label='Wavelength λ (m)')

# Panel (e): Pre/Post Oso Landslide Multiscale Complexity Change
ax = axes[1, 1]
ext_o = [oso_post.bounds[0]/1e3, oso_post.bounds[2]/1e3, oso_post.bounds[1]/1e3, oso_post.bounds[3]/1e3]
oso_hs = terrain_analysis.compute_hillshade(oso_post)
ax.imshow(oso_hs.data, cmap='gray', extent=ext_o, origin='upper')
vlim_o = float(np.nanpercentile(np.abs(delta_c), 98))
im4 = ax.imshow(delta_c, cmap='PuOr_r', vmin=-vlim_o, vmax=vlim_o, alpha=0.78, extent=ext_o, origin='upper')
ax.set_title('(e) Oso Δ complexity (post − pre)', fontweight='bold')
ax.set_xlabel('UTM Easting (km, Zone 10N)')
ax.set_ylabel('UTM Northing (km)')
plt.colorbar(im4, ax=ax, fraction=0.046, pad=0.04, label='ΔMean Complexity')

# Panel (f): Multi-Layer Swath Profile Across Sheep Mountain Anticline (A -> A')
ax = axes[1, 2]
mprof = cross_section.extract_multilayer_profile(
    {
        'Elevation': sheep_dem,
        'Energy_30m': sheep_cwt['energy_stack'].band('energy_30m'),
        'Energy_120m': sheep_cwt['energy_stack'].band('energy_120m'),
    },
    start_xy=p_start,
    end_xy=p_end,
    n_samples=200,
    swath_width_m=120.0,
)
d_m = mprof['distance_m']
ax.plot(d_m, mprof['layers']['Elevation']['mean'], 'k-', lw=2.0, label='Elevation (m)')
ax.fill_between(d_m, mprof['layers']['Elevation']['min'], mprof['layers']['Elevation']['max'], color='gray', alpha=0.22)
ax.set_xlabel('Distance Along Profile A–A\' (m)')
ax.set_ylabel('Elevation (m NAVD88)', color='k')
ax2 = ax.twinx()
ax2.plot(d_m, mprof['layers']['Energy_30m']['mean'], color='#d95f02', lw=1.6, label='CWT Energy (s=30 m)')
ax2.plot(d_m, mprof['layers']['Energy_120m']['mean'], color='#7570b3', lw=1.8, ls='--', label='CWT Energy (s=120 m)')
ax2.set_ylabel('Wavelet Energy |W(s)|/s', color='#7570b3')
ax.set_title("(f) Swath A–A': elevation vs. CWT energy", fontweight='bold')
ax.grid(True, ls=':', alpha=0.45)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=7.5)

for ax in axes.ravel()[:5]:
    plotting.use_plain_axes(ax)

fig.suptitle('pyGeoMapper — Rust 2D Mexican-hat CWT and multiscale topographic complexity', fontsize=14, fontweight='bold')
out_png = IMG_DIR / 'cwt_multiscale.png'
plotting.save_figure(fig, out_png)
print('Saved:', out_png)

Saved: image/cwt_multiscale.png


## 6. 2.5D Draped Perspective Visualization of Sheep Mountain Multiscale Complexity (`image/sheep_mtn_cwt_3d_drape.png`)
Finally, we drape the multiscale 2D CWT complexity surface over the 3DEP DEM using `viewer_3d.plot_draped_surface_matplotlib()` to visualize how the resistant Paleozoic limestone/sandstone inner canyon and Mesozoic hogback ridges stand out in 3D perspective.

In [7]:
fig_3d, ax_3d = viewer_3d.plot_3d_terrain_drape(
    sheep_dem,
    overlay=sheep_cwt['mean_complexity'],
    cmap='magma',
    vertical_exaggeration=2.2,
    elevation_deg=36.0,
    azimuth_deg=-55.0,
    stride=1,
    title='Sheep Mountain Anticline — 3DEP surface draped with multiscale CWT complexity',
    cbar_label='Mean Multiscale CWT Complexity',
)
plotting.save_figure(fig_3d, IMG_DIR / "sheep_mtn_cwt_3d_drape.png", max_width_px=1400)

PosixPath('image/sheep_mtn_cwt_3d_drape.png')